# CINF104 - Proyecto 1: Predicción GRD Hospital El Pino
## 01 - Análisis Exploratorio de Datos (EDA)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

print('Librerías cargadas correctamente')

## 1. Carga de datos

In [ ]:
df = pd.read_csv('../data/dataset_elpino.csv', sep=None, engine='python')
print(f'Filas: {len(df):,}')
print(f'Columnas: {len(df.columns)}')
df.head()

## 2. Estructura del dataset

In [ ]:
# Separar columnas por tipo
diag_cols = [c for c in df.columns if 'Diag' in c]
proced_cols = [c for c in df.columns if 'Proced' in c]

print(f'Columnas de diagnóstico: {len(diag_cols)}')
print(f'Columnas de procedimiento: {len(proced_cols)}')
print(f'Otras columnas: Edad, Sexo, GRD')

## 3. Calidad de datos - Valores nulos

In [ ]:
# Nulos en columnas clave
key_cols = ['Diag 01 Principal (cod+des)', 'Edad en años', 'Sexo (Desc)', 'GRD']
print('Nulos en columnas clave:')
print(df[key_cols].isna().sum())
print()

# Nulos por diagnóstico secundario
nulos_diag = df[diag_cols].isna().sum()
print('Completitud diagnósticos secundarios:')
for col, n in nulos_diag.items():
    pct_lleno = (1 - n/len(df)) * 100
    print(f'  {col[:30]}: {pct_lleno:.1f}% con datos')

## 4. Variable objetivo: GRD

In [ ]:
print(f'Clases únicas de GRD: {df["GRD"].nunique()}')
print()
print('Top 15 GRDs más frecuentes:')
top_grd = df['GRD'].value_counts().head(15)
print(top_grd)

In [ ]:
# Distribución de frecuencias por clase
conteos = df['GRD'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 20 clases
top20 = conteos.head(20)
axes[0].barh(range(len(top20)), top20.values)
axes[0].set_yticks(range(len(top20)))
axes[0].set_yticklabels([g[:40] for g in top20.index], fontsize=7)
axes[0].set_title('Top 20 GRDs más frecuentes')
axes[0].set_xlabel('Cantidad de pacientes')

# Distribución de frecuencias
axes[1].hist(conteos.values, bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Distribución de frecuencias por clase GRD')
axes[1].set_xlabel('Pacientes por clase')
axes[1].set_ylabel('Cantidad de clases')

plt.tight_layout()
plt.savefig('../reports/grd_distribucion.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Variables demográficas

In [ ]:
print('Distribución de Edad:')
print(df['Edad en años'].describe())
print()
print('Distribución de Sexo:')
print(df['Sexo (Desc)'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma edad
axes[0].hist(df['Edad en años'].dropna(), bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de Edad')
axes[0].set_xlabel('Edad (años)')
axes[0].set_ylabel('Pacientes')

# Pie chart sexo
sexo_counts = df['Sexo (Desc)'].value_counts()
axes[1].pie(sexo_counts.values, labels=sexo_counts.index, autopct='%1.1f%%', colors=['#ff9999','#66b3ff'])
axes[1].set_title('Distribución por Sexo')

plt.tight_layout()
plt.savefig('../reports/demograficos.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Diagnósticos: cuántos por paciente

In [ ]:
# Contar diagnósticos no nulos por fila
df['n_diagnosticos'] = df[diag_cols].notna().sum(axis=1)
df['n_procedimientos'] = df[proced_cols].notna().sum(axis=1)

print('Diagnósticos por paciente:')
print(df['n_diagnosticos'].describe())
print()
print('Procedimientos por paciente:')
print(df['n_procedimientos'].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['n_diagnosticos'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Diagnósticos por paciente')
axes[0].set_xlabel('Cantidad de diagnósticos')
axes[0].set_ylabel('Pacientes')

axes[1].hist(df['n_procedimientos'], bins=20, color='coral', edgecolor='white')
axes[1].set_title('Procedimientos por paciente')
axes[1].set_xlabel('Cantidad de procedimientos')
axes[1].set_ylabel('Pacientes')

plt.tight_layout()
plt.savefig('../reports/diag_proced_por_paciente.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Extracción de códigos CIE

In [ ]:
# Los diagnósticos vienen como "A41.8 - Descripción"
# Extraemos solo el código antes del " - "
def extraer_codigo(texto):
    if pd.isna(texto):
        return None
    return texto.split(' - ')[0].strip()

# Ejemplo
print('Ejemplo original:', df['Diag 01 Principal (cod+des)'].iloc[0])
print('Código extraído:', extraer_codigo(df['Diag 01 Principal (cod+des)'].iloc[0]))

In [ ]:
# Top diagnósticos principales más frecuentes
df['codigo_diag_principal'] = df['Diag 01 Principal (cod+des)'].apply(extraer_codigo)

print('Top 15 diagnósticos principales:')
print(df['codigo_diag_principal'].value_counts().head(15))